<a href="https://colab.research.google.com/github/nicolasramirezperilla/DataWave-Project/blob/master/Consolidado_BBDD_Filiales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##1) Instalar librerias y conexión al servidor.

In [ ]:
# Importing libraries
from google.colab import auth
from google.colab import files
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil import parser  # Import dateutil.parser for automatic date parsing

# Formatting for viewing tables
from google.colab import data_table
data_table.enable_dataframe_formatter()

# Authenticating Google Sheets
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

from gspread_dataframe import get_as_dataframe
import pandas as pd

gc = gspread.authorize(creds)

##2) Descargar información, definir parametros y transformación.

In [ ]:
# Nombre de la hoja a abrir
hoja = 'BBDD'

# Abre y convierte la hoja 'BBDD' en el libro 'FiduV2BBDD' a DataFrame
input_workbook_name_fidu = 'FiduV2BBDD'
workbook_fidu = gc.open(input_workbook_name_fidu)
worksheet_fidu = workbook_fidu.worksheet(hoja)
df_fidu = get_as_dataframe(worksheet_fidu)
df_fidu['Filial'] = 'Fiduciaria'

# Abre y convierte la hoja 'BBDD' en el libro 'ValBBDD' a DataFrame
input_workbook_name_val = 'ValBBDD'
workbook_val = gc.open(input_workbook_name_val)
worksheet_val = workbook_val.worksheet(hoja)
df_val = get_as_dataframe(worksheet_val)
df_val['Filial'] = 'Valores'

# Abre y convierte la hoja 'BBDD' en el libro 'SeguBBDD' a DataFrame
input_workbook_name_segu = 'SeguBBDD'
workbook_segu = gc.open(input_workbook_name_segu)
worksheet_segu = workbook_segu.worksheet(hoja)
df_segu = get_as_dataframe(worksheet_segu)
df_segu['Filial'] = 'Seguros'

# Abre y convierte la hoja 'BBDD' en el libro 'CSFBBDD' a DataFrame
input_workbook_name_csf = 'CSFBBDD'
workbook_csf = gc.open(input_workbook_name_csf)
worksheet_csf = workbook_csf.worksheet(hoja)
df_csf = get_as_dataframe(worksheet_csf)
df_csf['Filial'] = 'Comercializadora'

# Une todos los DataFrames en uno solo, rellenando las columnas faltantes con 0
df_total = pd.concat([df_fidu, df_val, df_csf,df_segu], ignore_index=True).fillna("")

# Diccionario para mapear los meses
meses = {
    '01': 'Enero', '02': 'Febrero', '03': 'Marzo', '04': 'Abril',
    '05': 'Mayo', '06': 'Junio', '07': 'Julio', '08': 'Agosto',
    '09': 'Septiembre', '10': 'Octubre', '11': 'Noviembre', '12': 'Diciembre'
}

# Extraer el mes y el año
df_total['Month'] = df_total['Date'].str[5:7].map(meses)
df_total['Year'] = df_total['Date'].str[:4]

In [ ]:
# Filtrar datos no nulos y vacíos
df_total = df_total[df_total['Month'].notna() & (df_total['Month'] != "")]
df_total = df_total[df_total['Year'].notna() & (df_total['Year'] != "")]
df_total = df_total[df_total['Filial'].notna() & (df_total['Filial'] != "")]
df_total = df_total.dropna(subset=['Month', 'Year', 'Filial'])

# Listado de columnas a convertir
columnas_a_convertir = ['Real', 'Proyec.','Real_o_Proyeccion', 'Ppto', 'Real_Acum_Año',
                        'Real_o_Proyec_Acum_Año', 'Ppto_Acum_Año',
                        'Real_Ant', 'Real_o_Proyec_Ant',
                        'Real_o_Proyeccion_12M', 'Real_Acum_Last_Year',
                        'Real_o_Proyec_Acum_Last_Year', 'MoM_Abs', 'MoM_Porc.',
                        'YoY_Abs', 'YoY_Porc.', 'Cump_Ppto_Abs',
                        'Cump_Ppto_Porc.', 'Cump_Ppto_Abs_Acum',
                        'Cump_Ppto_Porc_Acum', 'Rapel', 'RapelM']

# Convertir columnas a float, manejando errores
for col in columnas_a_convertir:
    df_total[col] = pd.to_numeric(df_total[col], errors='coerce')

#-----------------------------------------------------
# 1. Agrupación por mes, año, filial y Marca_Fecha para realizar cálculos
grouped = df_total.groupby(['Month', 'Year', 'Filial', 'Marca_Fecha'])

# 2. Función para calcular las nuevas filas
def generar_filas_de_calculo(grupo):
    calculos = []

    # Extraer valores únicos para las columnas relevantes
    month = grupo['Month'].iloc[0]
    year = grupo['Year'].iloc[0]
    filial = grupo['Filial'].iloc[0]
    marca_fecha = grupo['Marca_Fecha'].iloc[0]


    # Calcular Margen Bruto
    margen_bruto = {
        tipo: grupo.loc[grupo['Level9'].isin([
            'Margen de Intereses', 'Comisiones Netas', 'ROFs', 'Resto Ingresos Netos Ordinarios'
        ]), tipo].sum()
        for tipo in columnas_a_convertir
    }
    calculos.append([
        'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo',
        'Margen Bruto', month, year, filial, marca_fecha, *[margen_bruto[tipo] for tipo in columnas_a_convertir]
    ])

    # Calcular Gastos de Explotación
    gastos_explotacion = {
        tipo: grupo.loc[grupo['Level9'].isin([
            'Gastos de Personal', 'Gastos Generales', 'Tributos', 'Amortizaciones'
        ]), tipo].sum()
        for tipo in columnas_a_convertir
    }
    calculos.append([
        'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo',
        'Gastos de Explotación', month, year, filial, marca_fecha, *[gastos_explotacion[tipo] for tipo in columnas_a_convertir]
    ])

    # Calcular Margen Neto
    margen_neto = {
        tipo: margen_bruto[tipo] + gastos_explotacion[tipo] for tipo in columnas_a_convertir
    }
    calculos.append([
        'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo',
        'Margen Neto', month, year, filial, marca_fecha, *[margen_neto[tipo] for tipo in columnas_a_convertir]
    ])

    # Calcular Resultados de Explotación
    resultados_explotacion = {
        tipo: margen_neto[tipo] + grupo.loc[grupo['Level9'].isin([
            'Saneamiento Crediticio', 'Pérdida Deterioro Resto de Activos', 'Dotaciones a Provisiones'
        ]), tipo].sum()
        for tipo in columnas_a_convertir
    }
    calculos.append([
        'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo',
        'Resultados de Explotación', month, year, filial, marca_fecha, *[resultados_explotacion[tipo] for tipo in columnas_a_convertir]
    ])

    # Calcular BAI
    bai = {
        tipo: resultados_explotacion[tipo] + grupo.loc[grupo['Level9'] == 'Resultados No Ordinarios', tipo].sum()
        for tipo in columnas_a_convertir
    }
    calculos.append([
        'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo',
        'BAI', month, year, filial, marca_fecha, *[bai[tipo] for tipo in columnas_a_convertir]
    ])

    # Calcular BDI
    bdi = {
        tipo: bai[tipo] + grupo.loc[grupo['Level9'] == 'Impuesto Sociedades', tipo].sum()
        for tipo in columnas_a_convertir
    }
    calculos.append([
        'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo',
        'BDI', month, year, filial, marca_fecha, *[bdi[tipo] for tipo in columnas_a_convertir]
    ])

    # Calcular Resultado Atribuido
    resultado_atribuido = {
        tipo: bdi[tipo] + grupo.loc[grupo['Level9'] == 'Intereses Minoritarios', tipo].sum()
        for tipo in columnas_a_convertir
    }
    calculos.append([
        'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo', 'Cálculo',
        'Resultado Atribuido', month, year, filial, marca_fecha, *[resultado_atribuido[tipo] for tipo in columnas_a_convertir]
    ])

    return pd.DataFrame(calculos, columns=[
        'Category', 'Level1', 'Level2', 'Level3', 'Level4', 'Level5', 'Level7', 'Level8',
        'Level9', 'Month', 'Year', 'Filial', 'Marca_Fecha', *columnas_a_convertir
    ])

# 3. Aplicar función a cada grupo
nuevas_filas = grouped.apply(generar_filas_de_calculo).reset_index(drop=True)

# 4. Combinar el DataFrame original con las nuevas filas
df_total = pd.concat([df_total, nuevas_filas], ignore_index=True)

# 5. Actualizar la columna 'Date' para las filas con 'Category' igual a 'Cálculo'
df_total.loc[df_total['Category'] == 'Cálculo', 'Date'] = (
    df_total['Year'].astype(str) + "-" +
    df_total['Month'].astype(str).str.zfill(2) + "-30"
)


<ipython-input-15-8ddd51243ffc>:117: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nuevas_filas = grouped.apply(generar_filas_de_calculo).reset_index(drop=True)


## 3) Actualizar hojas de cálculo en Google Sheets

In [ ]:
input_workbook_name = 'TotalBBDD'
output_sheet_df_total = gc.open(input_workbook_name).worksheet('BBDD')
output_sheet_df_total.clear()
output_sheet_df_total.update([df_total.columns.values.tolist()] + df_total.fillna('').values.tolist())

<ipython-input-16-af739e6869ac>:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  output_sheet_df_total.update([df_total.columns.values.tolist()] + df_total.fillna('').values.tolist())


{'spreadsheetId': '1qmxyBRVaU0N8BCsJHImUfl_sLp5etCbLr4JDee816pQ',
 'updatedRange': 'BBDD!A1:AM22849',
 'updatedRows': 22849,
 'updatedColumns': 39,
 'updatedCells': 891111}